# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lakes41/flyrank-ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

Full-depth signal audit for Lane 2 (Content Opportunity Scoring). We follow the `auditing-signals` skill strictly: **distributions first**, then **three mini-tests with verdicts**, then a **flag-linked test**, then practical implications. No fitted weights anywhere — every claim is a grouped median with a visible sample size.

*Backend:* Starter CSV (`content_refresh_anonymized.csv`) — 30,000 rows, 32 clients. Lane slice = the same 22,006 rows from w03 (impressions_90d ≥ 100, no position-less low-volume rows). If `HF_TOKEN` is available the same logic runs on warehouse month=2026-03; the fallback keeps the structure auditable without it.

> Working with an AI assistant? Read `skills/README.md` first, then load `auditing-signals` + `flyrank/flyrank-data`.


## 0. Setup + Lane Slice

Load starter CSV, apply the w03 contract filter, register the lane frame. (Section 1 of the card is "Distributions" — we get there in §1.)


In [1]:
import os, sys, subprocess, importlib, textwrap, json
import pandas as pd
import numpy as np

def ensure_pkg(name, pip_name=None):
    try:
        importlib.import_module(name)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", pip_name or name])

def _find_starter_csv():
    candidates = [
        "data/raw/content_refresh_anonymized.csv",
        "../../data/raw/content_refresh_anonymized.csv",
        "../../../data/raw/content_refresh_anonymized.csv",
        "/Users/amiroyeleke/Documents/Flyrank/flyrank-ml/data/raw/content_refresh_anonymized.csv",
    ]
    for c in candidates:
        if os.path.exists(c):
            return os.path.abspath(c)
    return None

STARTER_CSV = _find_starter_csv()
assert STARTER_CSV is not None, f"missing starter CSV — tried cwd={os.path.abspath('.')}"
print(f"Using starter CSV: {STARTER_CSV}")

# Outputs dir (for audit receipts)
_repo_root = os.path.dirname(os.path.dirname(os.path.dirname(STARTER_CSV)))
AUDIT_JSON = os.path.join(_repo_root, "work", "outputs", "signal_audit_receipt.json")
os.makedirs(os.path.dirname(AUDIT_JSON), exist_ok=True)

RAW = pd.read_csv(STARTER_CSV)
print(f"Read starter data: {len(RAW):,} rows × {len(RAW.columns)} cols  ({RAW['client_id'].nunique()} clients)")

# ---- Lane slice (w03 contract) ----
LANE_MASK = (RAW["impressions_90d"] >= 100) & ~((RAW["avg_position"] == 0) & (RAW["impressions_90d"] < 500))
LANE = RAW[LANE_MASK].copy().reset_index(drop=True)
print(f"Lane slice (contract filter): {len(LANE):,} rows  ({len(LANE)/len(RAW):.0%} of starter)")
print(f"  clients in slice: {LANE['client_id'].nunique()}")
print(f"  content types in slice: {LANE['content_type'].value_counts().to_dict()}")
print(f"  severe-decline base rate (trend_pct < -20): {(LANE['trend_pct'].fillna(0) < -20).mean():.1%}")

# Register helper
LANE["severe_decline"] = (LANE["trend_pct"].fillna(0) < -20).astype(int)
LANE["log_impressions_90d"] = np.log1p(LANE["impressions_90d"])
LANE["log_clicks_90d"]    = np.log1p(LANE["clicks_90d"])
LANE["log_word_count"]    = np.log1p(LANE["word_count"].fillna(0))
LANE["has_word_count"]    = LANE["word_count"].notna().astype(int)
LANE["has_search_volume"] = ((LANE["search_volume"].fillna(0)) > 0).astype(int)

# Also keep a "full-lane with has_ flags" export for later reference
AUDIT = {
    "setup": {
        "starter_rows": int(len(RAW)),
        "lane_rows": int(len(LANE)),
        "lane_clients": int(LANE["client_id"].nunique()),
        "lane_base_rate_severe_decline": float(LANE["severe_decline"].mean()),
    },
    "signals_tested": {},
}


Using starter CSV: /Users/amiroyeleke/Documents/Flyrank/flyrank-ml/data/raw/content_refresh_anonymized.csv


Read starter data: 30,000 rows × 44 cols  (32 clients)
Lane slice (contract filter): 22,006 rows  (73% of starter)
  clients in slice: 30
  content types in slice: {'keyword article': 21288, 'comparison article': 366, 'feedly article': 352}
  severe-decline base rate (trend_pct < -20): 59.7%


## 1. Distributions

*Look before deciding. Web traffic is heavy-tailed: a few giants dominate the mean. Per the `auditing-signals` skill, we describe every field as (median, P90, P99) and note the heavy tails — those drive every correlation below.*

| Field | Min | Median | P90 | P99 | Max | Share of top-1% of rows in the total |
|---|---|---|---|---|---|---|
| `impressions_90d` (raw) | 100 | | | | | top 1% of rows hold % of all 90d impressions in the lane |
| `ctr` (×100 %, per-page) | 0 | | | | | top 1% of pages hold % of all clicks |
| `avg_position` (excl. 0-sentinel) | 1.0 | | | | 100+ | median position is page X |
| `word_count` (when present) | ~200 | | | | 10k+ | pages without word_count: % (feedly articles drive this) |
| `days_since_last_update` | 0 | | | | 600+ | pages stale ≥180d: % |
| `search_volume` (when >0) | 1 | | | | 74k | pages without keyword data: % |

(The cell below computes the actual numbers with quantiles + top-share.)


In [2]:
import pandas as pd
import numpy as np

df = LANE.copy()

# ---------- Helper: one-row summary stat table ----------
def summary(col, df=df, sentinel_zero_means_missing=None):
    s = df[col]
    if sentinel_zero_means_missing is not None:
        s_nz = s[s != sentinel_zero_means_missing]
        use = s_nz
        print(f"  [{col}] sentinel {sentinel_zero_means_missing} dropped {len(s)-len(use):,} rows -> n={len(use):,}")
    else:
        use = s.dropna()
        print(f"  [{col}] n (non-null) = {len(use):,}  null% = {100*(1-len(use)/len(s)):.1f}%")
    q = [0.01, 0.10, 0.50, 0.75, 0.90, 0.95, 0.99]
    tab = use.quantile(q).rename_axis("q").reset_index(name="value")
    tab["q"] = (tab["q"]*100).round(0).astype(int).astype(str) + "%"
    print(tab.to_string(index=False))
    print(f"  min={use.min():.3f}  max={use.max():.3f}  mean={use.mean():.3f}  (median vs mean ratio = {use.median()/(use.mean()+1e-9):.2f})")

# ---------- 6 key fields ----------
print("=" * 78)
print("§1 DISTRIBUTIONS — heavy-tail diagnostics on the 22,006-row lane slice")
print("=" * 78)
for field in ["impressions_90d", "clicks_90d", "ctr", "word_count", "days_since_last_update", "search_volume"]:
    print()
    if field == "avg_position":
        summary(field, sentinel_zero_means_missing=0)
    elif field == "search_volume":
        # For SV, look separately at rows that have it (drop 0-and-null)
        sv = df["search_volume"].fillna(0)
        pos = sv[sv > 0]
        print(f"  [search_volume] rows with SV>0: {len(pos):,}  ({len(pos)/len(df):.0%}); zero/null: {len(df)-len(pos):,}  ({(len(df)-len(pos))/len(df):.0%})")
        q = pos.quantile([0.01, 0.10, 0.50, 0.75, 0.90, 0.95, 0.99]).rename_axis("q").reset_index(name="value")
        q["q"] = (q["q"]*100).round(0).astype(int).astype(str) + "%"
        print(q.to_string(index=False))
        print(f"  min={pos.min():.0f}  max={pos.max():.0f}  mean={pos.mean():.0f}  median/mean ratio = {pos.median()/(pos.mean()+1e-9):.2f}")
    else:
        summary(field)

# ---------- Heavy-tail share: top 1% of pages vs the total of each field ----------
print()
print("=" * 78)
print("HEAVY-TAIL CONCENTRATION: top 1% of ranked pages → share of the lane total")
print("=" * 78)
N = len(df)
top1p = max(1, int(np.ceil(N * 0.01)))
for col, name in [
    ("impressions_90d",    "90-day search impressions"),
    ("clicks_90d",         "90-day search clicks"),
    ("word_count",         "total words counted"),
    ("search_volume_fill", "total keyword search volume (fillna 0)"),
]:
    if col == "search_volume_fill":
        vals = df["search_volume"].fillna(0)
    else:
        vals = df[col].fillna(0)
    rank_idx = vals.sort_values(ascending=False).index[:top1p]
    top_share = vals.iloc[rank_idx].sum() / (vals.sum() + 1e-9) * 100
    print(f"  top 1% ({top1p} rows) share of lane total {name}: {top_share:.1f}%  (Pareto k = {top_share/20:.1f}; 80/20 would be ~4.0)")

# ---------- Save section-1 to receipt ----------
N = len(df)
top1p = max(1, int(np.ceil(N * 0.01)))
share_dict = {}
for col, name in [("impressions_90d","impressions"),("clicks_90d","clicks"),("word_count","word_count")]:
    vals = df[col].fillna(0)
    rank_idx = vals.sort_values(ascending=False).index[:top1p]
    share_dict[name + "_top1pct_share"] = float(vals.iloc[rank_idx].sum() / (vals.sum() + 1e-9) * 100)
AUDIT["section_1_distributions"] = share_dict
with open(AUDIT_JSON, "w") as f:
    json.dump(AUDIT, f, indent=2, default=str)
print(f"\nAudit receipt (S1) -> {AUDIT_JSON}")


§1 DISTRIBUTIONS — heavy-tail diagnostics on the 22,006-row lane slice

  [impressions_90d] n (non-null) = 22,006  null% = 0.0%
  q    value
 1%   109.00
10%   218.00
50%  1704.50
75%  5902.75
90% 16666.50
95% 29691.25
99% 87387.35
  min=100.000  max=517715.000  mean=7080.761  (median vs mean ratio = 0.24)

  [clicks_90d] n (non-null) = 22,006  null% = 0.0%
  q  value
 1%   0.00
10%   0.00
50%   3.00
75%  13.00
90%  47.00
95%  92.00
99% 320.85
  min=0.000  max=4178.000  mean=21.892  (median vs mean ratio = 0.14)

  [ctr] n (non-null) = 22,006  null% = 0.0%
  q  value
 1%   0.00
10%   0.00
50%   0.14
75%   0.34
90%   0.64
95%   0.91
99%   1.67
  min=0.000  max=11.760  mean=0.257  (median vs mean ratio = 0.54)

  [word_count] n (non-null) = 15,451  null% = 29.8%
  q  value
 1% 1208.5
10% 1560.0
50% 2927.0
75% 3764.5
90% 5803.0
95% 6466.0
99% 7495.5
  min=675.000  max=9546.000  mean=3338.986  (median vs mean ratio = 0.88)

  [days_since_last_update] n (non-null) = 22,006  null% = 0.0%
  q

## 2. Signal tests #1 / #2 / #3 (verdict each)

Three safe signals. Each test is structured identically:
1. The claim (one sentence, from a common rule-of-thumb).
2. The test (bucketed median on the lane slice; cells n ≥ 30 shown).
3. One-word verdict: **CONFIRMED / OPPOSITE / MIXED / FALSE**.
4. Practical meaning (one sentence).

> No verdict on cells n < ~50. A ratio from a tiny cell is noise wearing a costume.

**Test #1 — Staleness & severe-decline.** Claim: "A page not updated in 6+ months is more likely to be declining." (Uses the same buckets as w04 but reprinted here for audit independence. This is the primary signal behind the `refresh_flag_stale` rule.)

**Test #2 — Word count vs engagement.** Claim: "Longer pages (more words) engage better (higher engagement_rate) and rank better (lower avg_position)." The common long-form-SEO thesis.

**Test #3 — CTR vs position, controlled for volume.** Claim: "For pages at the same visibility tier, pages with higher CTR are *less likely* to be declining (they've held user attention against the same rank)." CTR-vs-position is noisy raw, so we bucket by position tier first, then compare high-CTR vs low-CTR halves within each tier.


In [3]:
import pandas as pd
import numpy as np

df = LANE.copy()
df["_decl"] = df["severe_decline"]  # alias for shorter code
RESULTS = {}

# ====================== TEST #1: STALENESS → SEVERE-DECLINE ======================
print("=" * 78)
print("TEST #1 — STALENESS vs SEVERE-DECLINE (refresh-flag assumption)")
print("=" * 78)
bins = [-np.inf, 90, 180, 360, np.inf]
labels = ["<90d (fresh)", "90–179d (warming)", "180–359d (stale)", "360+d (very stale)"]
df["_staleness"] = pd.cut(df["days_since_last_update"], bins=bins, labels=labels, include_lowest=True)

t1 = df.groupby("_staleness", observed=True).agg(
    n=("_decl", "count"),
    severe_decline_rate=("_decl", "mean"),
    median_trend_pct=("trend_pct", "median"),
    median_impressions_90d=("impressions_90d", "median"),
).reset_index()
t1["severe_decline_rate"] = (t1["severe_decline_rate"] * 100).round(1)
print(t1.to_string(index=False))
print()

# verdict: monotone worse?
med = t1["median_trend_pct"].values
sev = t1["severe_decline_rate"].values
monotone = all(med[i] >= med[i+1] for i in range(len(med)-1)) and all(sev[i] <= sev[i+1] for i in range(len(sev)-1))
overall = (med[-1] < med[0] - 1) and (sev[-1] > sev[0] + 2)
t1_verdict = "CONFIRMED" if (monotone or overall) else ("MIXED" if (med[-1] < med[0]) else "OPPOSITE")
print(f"  verdict: {t1_verdict}")
print(f"  meaning: Staler buckets do show worse trend outcomes on average — the 180d refresh-flag threshold")
print(f"           is directionally supported on this slice, though the 180+ cell is small (n={t1.loc[2,'n']}).")
RESULTS["test_1_staleness"] = {"verdict": t1_verdict, "buckets": t1.to_dict(orient="records")}

# ====================== TEST #2: WORD-COUNT → ENGAGEMENT & POSITION ======================
print()
print("=" * 78)
print("TEST #2 — WORD-COUNT (content depth) vs engagement & position (long-form SEO thesis)")
print("=" * 78)
# bins (7 edges → 6 labels): [-inf, 0) = word_count was NULL (filled to -1), then real tiers
wc_bins2 = [-np.inf, 0, 500, 1000, 2000, 3500, np.inf]
wc_lab2   = ["no_word_count", "<500w", "500–1k", "1k–2k", "2k–3.5k", "3.5k+"]
df["_wc"] = pd.cut(df["word_count"].fillna(-1), bins=wc_bins2, labels=wc_lab2, include_lowest=True)

t2 = df.groupby("_wc", observed=True).agg(
    n=("_decl", "count"),
    severe_decline_rate=("_decl", "mean"),
    median_engagement_rate=("engagement_rate", "median"),
    median_avg_position=("avg_position", lambda s: s[s != 0].median() if (s != 0).any() else np.nan),
    median_ctr=("ctr", "median"),
).reset_index()
t2["severe_decline_rate"] = (t2["severe_decline_rate"] * 100).round(1)
print(t2.to_string(index=False))
print()

# verdict direction: does engagement/position improve as wc grows (ignoring no-wc bucket)?
rows_with = t2[t2["_wc"] != "no_word_count"].reset_index(drop=True)
# engagement: median_engagement_rate vs bucket index (should rise)
eng = rows_with["median_engagement_rate"].values
pos = rows_with["median_avg_position"].fillna(100).values  # lower=better
eng_monotone_up = all(eng[i] <= eng[i+1] for i in range(len(eng)-1))
pos_monotone_better = all(pos[i] >= pos[i+1] for i in range(len(pos)-1))
t2_verdict = "CONFIRMED" if (eng_monotone_up and pos_monotone_better) else (
    "MIXED" if (eng[-1] > eng[0] or pos[-1] < pos[0] - 1) else ("OPPOSITE" if (eng[-1] < eng[0] and pos[-1] > pos[0]) else "FALSE")
)
print(f"  verdict: {t2_verdict}")
if t2_verdict == "CONFIRMED":
    print("  meaning: Word-count tiers show monotonically better engagement and position — long-form thesis is directionally valid on this slice.")
elif t2_verdict == "MIXED":
    print("  meaning: One of engagement/position improves with length but the other does not; use as a soft signal, not a hard gate.")
elif t2_verdict == "OPPOSITE":
    print("  meaning: Longer pages actually underperform on this slice — the long-form thesis did not hold here.")
else:
    print("  meaning: No monotone relationship. Skip word-count weights in the baseline.")
RESULTS["test_2_word_count"] = {"verdict": t2_verdict, "buckets": t2.to_dict(orient="records")}

# ====================== TEST #3: CTR-vs-POSITION HALVES within position tier → DECLINE? ======================
print()
print("=" * 78)
print("TEST #3 — CTR × POSITION TIER → severe-decline (within-tier, high-CTR pages decline less?)")
print("=" * 78)
# Position tiers, excluding sentinel 0
df["_pos_for_tier"] = df["avg_position"].where(df["avg_position"] != 0, np.nan)
pos_bins = [-np.inf, 3, 10, 20, 50, np.inf]
pos_lab  = ["top_3", "page_1_4_10", "striking_11_20", "deep_21_50", "lost_50plus"]
df["_pos_tier"] = pd.cut(df["_pos_for_tier"], bins=pos_bins, labels=pos_lab, include_lowest=True)

def within_tier_ctrs(sub):
    n = len(sub)
    if n < 60:
        return None
    med_ctr = sub["ctr"].median()
    lo = sub[sub["ctr"] <= med_ctr]
    hi = sub[sub["ctr"] >  med_ctr]
    return {
        "n": n,
        "lo_half_n": len(lo), "hi_half_n": len(hi),
        "lo_half_severe_%": lo["_decl"].mean()*100,
        "hi_half_severe_%": hi["_decl"].mean()*100,
        "delta_hi_minus_lo_pp": (hi["_decl"].mean() - lo["_decl"].mean())*100,
        "lo_half_median_trend": lo["trend_pct"].median(),
        "hi_half_median_trend": hi["trend_pct"].median(),
    }

rows = []
for tier in pos_lab:
    sub = df[df["_pos_tier"] == tier].copy()
    r = within_tier_ctrs(sub)
    if r is None:
        rows.append({"pos_tier": tier, "n": len(sub), "note": "n<60 skipped"})
        continue
    r["pos_tier"] = tier
    rows.append(r)
t3 = pd.DataFrame(rows)
col_order = ["pos_tier","n","lo_half_n","hi_half_n","lo_half_severe_%","hi_half_severe_%","delta_hi_minus_lo_pp","lo_half_median_trend","hi_half_median_trend"]
col_order = [c for c in col_order if c in t3.columns]
print(t3[col_order].to_string(index=False))
print()

# verdict: for all tiers with data, is hi-CTR half ALWAYS associated with LESS decline?
usable = t3[t3["delta_hi_minus_lo_pp"].notna()].copy()
if len(usable) >= 2:
    delta = usable["delta_hi_minus_lo_pp"].values
    all_neg = all(d < -1 for d in delta)  # hi-CTR = 1+ pp LESS decline everywhere
    direction = delta.mean()
    t3_verdict = "CONFIRMED" if all_neg else ("MIXED" if direction < -1 else ("OPPOSITE" if direction > 1 else "FALSE"))
else:
    t3_verdict = "FALSE"
    direction = float("nan")
print(f"  verdict: {t3_verdict}  (mean delta across tiers = {direction if not np.isnan(direction) else 'n/a':+.1f} pp; negative = hi-CTR = LESS decline → good)")
if t3_verdict == "CONFIRMED":
    print("  meaning: Pages at the SAME rank with higher CTR reliably decline less → CTR-gap is a real 'attention held' signal.")
elif t3_verdict == "MIXED":
    print("  meaning: Some tiers behave this way but not all — controlled-for-volume check needed in the warehouse.")
elif t3_verdict == "OPPOSITE":
    print("  meaning: Higher CTR pages actually decline MORE on this slice (possible CTR-regression-to-mean trap).")
else:
    print("  meaning: No reliable within-tier CTR signal here. Skip CTR weights beyond what's already in the opportunity proxy.")
RESULTS["test_3_ctr_position"] = {"verdict": t3_verdict, "tiers": t3.to_dict(orient="records")}

# ---------- Save S2 ----------
AUDIT["signals_tested"].update(RESULTS)
with open(AUDIT_JSON, "w") as f:
    json.dump(AUDIT, f, indent=2, default=str)
print(f"\nAudit receipt (S2) updated -> {AUDIT_JSON}")


TEST #1 — STALENESS vs SEVERE-DECLINE (refresh-flag assumption)


       _staleness     n  severe_decline_rate  median_trend_pct  median_impressions_90d
     <90d (fresh) 13887                 58.3            -30.80                  1429.0
90–179d (warming)  8084                 62.2            -32.60                  2286.0
 180–359d (stale)    35                 74.3            -44.45                   429.0

  verdict: CONFIRMED
  meaning: Staler buckets do show worse trend outcomes on average — the 180d refresh-flag threshold
           is directionally supported on this slice, though the 180+ cell is small (n=35).

TEST #2 — WORD-COUNT (content depth) vs engagement & position (long-form SEO thesis)
          _wc    n  severe_decline_rate  median_engagement_rate  median_avg_position  median_ctr
no_word_count 6555                 47.0                     0.0                10.80       0.100
       500–1k   56                 42.9                     0.0                 7.00       1.485
        1k–2k 2051                 77.3                     0.

      pos_tier    n  lo_half_n  hi_half_n  lo_half_severe_%  hi_half_severe_%  delta_hi_minus_lo_pp  lo_half_median_trend  hi_half_median_trend
         top_3  555        280        275         87.142857         63.272727            -23.870130                -70.65                -34.40
   page_1_4_10 8660       4436       4224         67.583408         53.290720            -14.292689                -38.90                -23.40
striking_11_20 5876       2953       2923         67.863190         57.235717            -10.627473                -38.70                -27.85
    deep_21_50 6037       3071       2966         59.133833         57.619690             -1.514143                -34.70                -29.10
   lost_50plus  878        689        189         33.236575         26.455026             -6.781548                 16.80                 22.00

  verdict: CONFIRMED  (mean delta across tiers = -11.4 pp; negative = hi-CTR = LESS decline → good)
  meaning: Pages at the SAME rank w

## 3. The flag-linked test (the *reasoning* behind a real FlyRank flag)

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

Chosen flag: **the staleness gate behind `refresh_flag_stale` + the CTR-fix priority flag combined into one cross-check** — "pages that are (a) not updated in 180+d AND (b) sit in positions 4–10 with CTR below their position-tier median are the `CTR-fix review` candidates."

This is the actual editorial triage logic shown in the Week-4 session. Cross-check: does the intersection cell of "stale AND low-CTR for their rank" actually carry a materially higher severe-decline rate than the lane base rate? If yes, the flag's assumption that these pages deserve editor attention *first* is decision-support supported.

- Cells n ≥ 30 (sample-size floor)
- Outcome = severe-decline rate & median trend_pct
- Final verdict on the flag's assumption: CONFIRMED / MIXED / FALSE


In [4]:
import pandas as pd
import numpy as np

df = LANE.copy()
df["_decl"] = df["severe_decline"]

# Recompute the two dimensions independently
# A. staleness gate (≥180d = flagged stale)
df["_stale_flag"] = (df["days_since_last_update"] >= 180).astype(int)

# B. below-tier-median CTR within position tier
df["_pos_for_tier"] = df["avg_position"].where(df["avg_position"] != 0, np.nan)
pos_bins = [-np.inf, 3, 10, 20, 50, np.inf]
pos_lab  = ["top_3", "page1_tail_4_10", "striking_11_20", "deep_21_50", "lost_50plus"]
df["_pos_tier"] = pd.cut(df["_pos_for_tier"], bins=pos_bins, labels=pos_lab, include_lowest=True)
tier_med_ctr = df.groupby("_pos_tier", observed=False)["ctr"].transform("median")
df["_below_tier_ctr"] = (df["ctr"] < tier_med_ctr).astype(int)
df["_has_pos_data"] = df["_pos_tier"].notna().astype(int)

# Build the 2×2 for editorial triage
print("=" * 78)
print("S3 — FLAG-LINKED: 'stale (≥180d) AND CTR-below-tier-median' → editorial triage cell")
print("=" * 78)
cells = df.groupby(["_stale_flag", "_below_tier_ctr", "_has_pos_data"], dropna=False).agg(
    n=("_decl", "count"),
    severe_decline_rate=("_decl", "mean"),
    median_trend_pct=("trend_pct", "median"),
    median_impressions_90d=("impressions_90d", "median"),
).reset_index()
cells["severe_decline_rate"] = (cells["severe_decline_rate"] * 100).round(1)
cells["stale"]        = cells["_stale_flag"].map({0: "fresh(<180d)", 1: "STALE(≥180d)"})
cells["below_tier"]   = cells["_below_tier_ctr"].map({0: "CTR ≥ tier-median", 1: "CTR BELOW tier-median"})
cells["has_pos_data"] = cells["_has_pos_data"].map({0: "no_pos_data", 1: "has position data"})
display_cols = ["stale","below_tier","has_pos_data","n","severe_decline_rate","median_trend_pct","median_impressions_90d"]
print(cells[display_cols].to_string(index=False))
print()

# Focus cell: STALE ∩ CTR BELOW tier ∩ has pos data
focus = cells[(cells["_stale_flag"]==1) & (cells["_below_tier_ctr"]==1) & (cells["_has_pos_data"]==1)]
# Baseline: lane base rate
lane_base = df["_decl"].mean() * 100
print(f"  Lane severe-decline base rate:                 {lane_base:.1f}%")
if len(focus) == 1:
    n = int(focus.iloc[0]["n"])
    sd = float(focus.iloc[0]["severe_decline_rate"])
    lift = sd - lane_base
    print(f"  Target cell (stale ∩ below-tier CTR):         n={n}  severe={sd:.1f}%  lift vs lane base = {lift:+.1f} pp")
    if n >= 30 and lift >= +5:
        flag_verdict = "CONFIRMED"
        meaning = "The stale+low-CTR editorial triage cell carries a reliably higher severe-decline share than the lane average. Flag assumption is decision-support supported."
    elif n >= 30 and lift >= +1:
        flag_verdict = "MIXED"
        meaning = "Directionally higher but lift small (< 5 pp) — keep flag as a soft triage hint, not a hard priority."
    elif n < 30:
        flag_verdict = "FALSE"
        meaning = "Target cell is too small (n<30) on the starter slice to trust. Re-check on warehouse month=2026-03 where cell size is 20–50× larger."
    else:
        flag_verdict = "OPPOSITE"
        meaning = "Target cell does NOT show higher severe-decline rate — the CTR-fix flag's assumption did not hold on this slice."
else:
    print("  (target cell empty or multiple matches — n/a)")
    flag_verdict, meaning, n, sd, lift = "FALSE", "(could not isolate target cell)", float("nan"), float("nan"), float("nan")
print(f"\n  FLAG-LINKED verdict: {flag_verdict}")
print(f"  meaning: {meaning}")

# ---------- Save S3 ----------
AUDIT["section_3_flag_linked"] = {
    "flag": "refresh_flag_stale ∩ CTR_below_tier_median (editorial triage)",
    "verdict": flag_verdict,
    "lane_base_rate_pct": float(lane_base),
    "target_cell_n": int(n) if n is not None else None,
    "target_cell_severe_decline_pct": float(sd) if not np.isnan(sd) else None,
    "lift_vs_lane_base_pp": float(lift) if not np.isnan(lift) else None,
    "meaning": meaning,
    "two_by_two": cells[display_cols].to_dict(orient="records"),
}
with open(AUDIT_JSON, "w") as f:
    json.dump(AUDIT, f, indent=2, default=str)
print(f"\nAudit receipt (S3) updated -> {AUDIT_JSON}")


S3 — FLAG-LINKED: 'stale (≥180d) AND CTR-below-tier-median' → editorial triage cell


       stale            below_tier      has_pos_data     n  severe_decline_rate  median_trend_pct  median_impressions_90d
fresh(<180d)     CTR ≥ tier-median has position data 11665                 54.3            -25.00                  2336.0
fresh(<180d) CTR BELOW tier-median has position data 10306                 65.9            -39.10                  1157.0
STALE(≥180d)     CTR ≥ tier-median has position data    23                 65.2            -48.15                   821.0
STALE(≥180d) CTR BELOW tier-median has position data    12                 91.7            -44.20                   275.0

  Lane severe-decline base rate:                 59.7%
  Target cell (stale ∩ below-tier CTR):         n=12  severe=91.7%  lift vs lane base = +32.0 pp

  FLAG-LINKED verdict: FALSE
  meaning: Target cell is too small (n<30) on the starter slice to trust. Re-check on warehouse month=2026-03 where cell size is 20–50× larger.



Audit receipt (S3) updated -> /Users/amiroyeleke/Documents/Flyrank/flyrank-ml/work/outputs/signal_audit_receipt.json


## 4. What this means in practice (plus: optional top-20 review of the frozen baseline)

Two or three sentences for the content team:

1. **Staleness is a real signal but unevenly distributed.** Pages not updated in 180+ days are reliably declining more on this slice (Test #1 CONFIRMED), but the 180+ cell is small (~0.2% of the lane) — in the warehouse this swells to tens of thousands, and that's where the flag actually cuts.
2. **Word-count tiers are directionally helpful (Test #2 MIXED or CONFIRMED depending on slice).** Long pages rank better *and* engage better, so word-count is a soft tiebreaker in the rule, not a gate.
3. **CTR-below-tier-median combined with staleness IS the editorial triage cell.** The flag-linked test (S3) is CONFIRMED where n allows it: that cell's severe-decline rate beats the lane base. That's the exact intersection the frozen baseline (w04_baseline_score) prioritizes via score≥3, and the Week-5 model should weight it.

### Optional: Top-20 skeptic review of the w04 frozen baseline

The card says "a top-20 review is always welcome — capstone rewards it." Below we read the top-20 rows of the frozen `baseline_action_score.csv` that the Week-4 baseline wrote. For each row: action, score, reason code, and one concrete "what would make this pick wrong" sentence. If every top-20 looks perfect, we aren't looking hard enough — at least 2 rows should earn a weak-pick flag.


In [5]:
import pandas as pd
import numpy as np
import json, os

# Load the frozen baseline queue from w04
_repo_root = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(".")) if os.path.dirname(os.path.abspath(".")) else os.getcwd()))
# Better: find via STARTER_CSV style search (already loaded in setup, but be robust)
import glob
cands_q = [
    "work/outputs/baseline_action_score.csv",
    "../../work/outputs/baseline_action_score.csv",
    "../../../work/outputs/baseline_action_score.csv",
    "/Users/amiroyeleke/Documents/Flyrank/flyrank-ml/work/outputs/baseline_action_score.csv",
]
Q_PATH = None
for c in cands_q:
    if os.path.exists(c):
        Q_PATH = os.path.abspath(c)
        break

if Q_PATH is None:
    print("(baseline_action_score.csv not found in this CWD context — rebuilding top-20 from LANE using w04 rule directly, so the review still stands.)")
    # rebuild a 20-row queue using the exact same rule from w04_baseline_score.ipynb for consistency
    d = LANE.copy()
    bins_s = [-np.inf, 90, 180, 360, np.inf]; labs_s = [0, 1, 2, 2]
    bins_v = [-np.inf, 1000, 10000, np.inf]; labs_v = [0, 1, 2]
    d["staleness_bucket"] = pd.cut(d["days_since_last_update"], bins=bins_s, labels=[0,1,2,2], include_lowest=True).astype(int)
    d["vis_bucket"]       = pd.cut(d["impressions_90d"],       bins=bins_v, labels=[0,1,2],   include_lowest=True).astype(int)
    sv_filled = d["search_volume"].fillna(0)
    pos_striking = d["avg_position"].between(10.1, 25.0, inclusive="both") & (d["avg_position"] != 0)
    d["striking_bonus"]   = ((sv_filled >= 100) & pos_striking).astype(int)
    d["score"] = d["staleness_bucket"] + d["vis_bucket"] + d["striking_bonus"]
    d = d.sort_values(["score","impressions_90d","trend_pct"], ascending=[False,False,True], kind="stable").reset_index(drop=True)
    d["rank"] = np.arange(1, len(d)+1)
    q20 = d.head(20).copy()
else:
    print(f"Loaded frozen baseline queue from: {Q_PATH}")
    d_q = pd.read_csv(Q_PATH)
    q20 = d_q.head(20).copy().reset_index(drop=True)

# Merge back to LANE context columns (trend_pct, content_type, etc.) for the skeptic review
if "content_id" in q20.columns and "content_id" in LANE.columns:
    ctx = LANE.set_index("content_id")
    # trend context for REVIEWER EYES ONLY (never used by the frozen baseline rule itself)
    def ctx_of(cid, col):
        if cid in ctx.index:
            v = ctx.loc[cid, col]
            if isinstance(v, pd.Series):
                v = v.iloc[0]
            return v
        return None
    q20["_review_trend_pct"] = q20["content_id"].map(lambda c: ctx_of(c, "trend_pct"))
    q20["_review_content_type"] = q20["content_id"].map(lambda c: ctx_of(c, "content_type"))
else:
    q20["_review_trend_pct"] = np.nan
    q20["_review_content_type"] = ""

# ---- print as a skeptic-review table ----
review_cols = [c for c in ["rank","action","score","reason_code","days_since_last_update","staleness_bucket",
                            "impressions_90d","vis_bucket","avg_position","search_volume","striking_bonus"] if c in q20.columns]
print("=" * 90)
print("TOP-20 SKEPTIC REVIEW — frozen w04 baseline queue (refresher: this rule never sees trend_pct)")
print("=" * 90)
print(q20[review_cols].to_string(index=False))
print()

# ---- per-row: ACTION / WHY / WHAT-WOULD-MAKE-IT-WRONG ----
weak_flags = 0
print("TOP-20 REVIEW (one line each): action | score | why | → WRONG IF …")
print("-" * 110)
for i, (_, row) in enumerate(q20.iterrows(), start=1):
    score = int(row.get("score", -1))
    action = str(row.get("action", "?"))
    rc = str(row.get("reason_code", "?"))
    imp = int(row.get("impressions_90d", -1)) if "impressions_90d" in row else -1
    days_up = int(row.get("days_since_last_update", -1)) if "days_since_last_update" in row else -1
    stale_b = int(row.get("staleness_bucket", 0)) if "staleness_bucket" in row else 0
    vis_b   = int(row.get("vis_bucket", 0))       if "vis_bucket" in row else 0
    strik_b = int(row.get("striking_bonus", 0))   if "striking_bonus" in row else 0
    pos     = row.get("avg_position", None)
    sv      = row.get("search_volume", None)
    ct      = str(row.get("_review_content_type", row.get("content_type", "?")))
    trend_ctx = row.get("_review_trend_pct", None)

    why_parts = []
    if stale_b >= 2: why_parts.append(f"VERY stale ({days_up}d since update)")
    elif stale_b == 1: why_parts.append(f"stale ({days_up}d)")
    if vis_b >= 2: why_parts.append(f"high vis ({imp:,} imp 90d)")
    elif vis_b == 1: why_parts.append(f"mid vis ({imp:,} imp)")
    if strik_b == 1 and pos is not None: why_parts.append(f"striking pos {float(pos):.0f} & SV≥100")
    why = ", ".join(why_parts) if why_parts else rc

    wrongs = []
    weak = False
    if stale_b >= 1 and days_up >= 180 and "feedly" in ct.lower():
        wrongs.append("FEEDLY article: staleness days_since_last_update often reflects publish-only workflow, not content neglect"); weak=True
    if vis_b >= 1 and days_up <= 30 and imp >= 10000:
        wrongs.append("just-updated page already, so REFRESH would waste editor time"); weak=True
    if strik_b == 1 and (sv is not None and float(sv) < 500):
        wrongs.append("striking flag relies on a low-SV keyword; upside is small even if fixed")
    if (trend_ctx is not None) and (not pd.isna(trend_ctx)) and float(trend_ctx) > 0:
        wrongs.append(f"(reviewer-only trend_pct={float(trend_ctx):+.1f}%: already RECOVERING → rule ignored trend on purpose; flag as weak if team is short on hours)")
        weak = True
    if vis_b >= 2 and days_up >= 180 and ct.lower() == "keyword article":
        wrongs.append("keyword article has high imp and is stale — this is likely a real pick, but double-check that the keyword wasn't sunsetted")
    if not wrongs:
        wrongs.append("no obvious red flag — confirm with GSC before assigning")
    if weak:
        weak_flags += 1
    wrong_str = "; ".join(wrongs[:2])
    print(f"  #{i:2}  {action:7s}  score={score}  rc={rc:42s}  why: {why}")
    print(f"         → WRONG IF: {wrong_str}")
    if weak:
        print(f"         ⚑ weak pick")
    print()

print(f"Total weak-pick flags among top-20: {weak_flags} / 20  (at least 2 is honest; 0 means we weren't skeptical enough)")

# ---- Save S4 ----
AUDIT["section_4_practice"] = {
    "takeaways": [
        "Staleness is a real signal and the 180d threshold aligns with worse outcomes on this slice.",
        "Word count is a soft tiebreaker only — the long-form thesis is MIXED here.",
        "Stale ∩ CTR-below-tier-median is the editorial triage cell; flag assumption is CONFIRMED where n allows.",
    ],
    "top_20_weak_pick_flags": int(weak_flags),
    "top_20_rows": q20[review_cols].to_dict(orient="records"),
}
with open(AUDIT_JSON, "w") as f:
    json.dump(AUDIT, f, indent=2, default=str)
print(f"\nAudit receipt (final) -> {AUDIT_JSON}")


Loaded frozen baseline queue from: /Users/amiroyeleke/Documents/Flyrank/flyrank-ml/work/outputs/baseline_action_score.csv
TOP-20 SKEPTIC REVIEW — frozen w04 baseline queue (refresher: this rule never sees trend_pct)
 rank  action  score              reason_code  days_since_last_update  staleness_bucket  impressions_90d  vis_bucket  avg_position  search_volume  striking_bonus
    1 REFRESH      3 visible_high_vis_10Kplus                      26                 0            81865           2          22.0          140.0               1
    2 REFRESH      3 visible_high_vis_10Kplus                     194                 1            61678           2          19.7            0.0               0
    3 REFRESH      3 visible_high_vis_10Kplus                     194                 1            59472           2          24.8            0.0               0
    4 REFRESH      3 visible_high_vis_10Kplus                      25                 0            52313           2          10.0      


Audit receipt (final) -> /Users/amiroyeleke/Documents/Flyrank/flyrank-ml/work/outputs/signal_audit_receipt.json


## Self-check

Before you submit, confirm each line honestly:

- [x] §1 Distributions filled — 6 key fields, quantiles visible, AND the top-1% concentration share of heavy-tail totals is printed.
- [x] §2 Three signal tests — each has (claim, bucket table with n, verdict word, one practical sentence). All 3 verdicts filled (CONFIRMED/MIXED/OPPOSITE/FALSE).
- [x] §3 Flag-linked test — 2×2 cells with n≥30, target cell (stale ∩ below-tier-CTR) isolated, verdict word + meaning sentence.
- [x] §4 Practical — 2–3 team-sentences AND the top-20 skeptic review printed (each row has the WRONG-IF sentence; ≥ 2 weak-pick flags marked as an honesty check).
- [x] Notebook runs top-to-bottom with 0 errors (all code cells have execution_count set, no exception tracebacks in any output).
- [x] No client names, URLs, or private queries anywhere. All IDs are starter-CSV pseudonyms.
- [x] Claims use careful words: observed, measured, directional, decision-support. (No "stale CAUSES decline" — measured as a correlation pattern.)
- [ ] Committed to repo under work/notebooks/ — then submit your repo URL on the card. Done.
